# Weather as Training Data

Companion notebook. Requires a FIT file and an internet connection
(to query the Open-Meteo API).

## Installation

The `openmeteo_requests` package is needed for the API call.

In [ ]:
# Uncomment and run if needed
# !pip install fitparse pandas numpy plotly openmeteo_requests

## Imports

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from fitparse import FitFile
import openmeteo_requests

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook"
print("Imports OK.")


Imports OK.


---
## User parameters

**Edit this cell only.**

In [ ]:
FIT_PATH = "/home/gsainton/01_CODES/Twinity/data/export_nolio_Greg_Sainton/SainteLyon_2025-11-29 23:31:00.fit"       # path to your .fit file
RAVITO_KM  = [19.2, 34.0, 45.0, 58.8, 65.4]
RAVITO_NOM = ['St Christo', 'Ste Catherine', 'St Genou', 'Soucieu', 'Chaponost']
TIMEZONE   = "Europe/Paris"
RACE_END   = None            # set only for multi-day races (YYYY-MM-DD)

EXPORT_FOR_BLOG = True
EXPORT_DIR = os.path.join(
    "assets", "img", "blog", "2026-04_weather"
)
if EXPORT_FOR_BLOG:
    os.makedirs(EXPORT_DIR, exist_ok=True)


---
## 1. Load FIT and extract GPS centroid

In [ ]:
from trail_analysis import load_fit, clean_df

df_raw = load_fit(FIT_PATH)
df, alt_col = clean_df(df_raw)

# Extract centroid for API call
lat_center = df["lat"].median()
lon_center = df["lon"].median()
print(f"GPS centroid: {lat_center:.4f}, {lon_center:.4f}")


GPS centroid: 45.6071, 4.5961


---
## 2. Fetch ERA5-Land data

In [ ]:
_ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
_HOURLY_VARS = [
    "temperature_2m", "relative_humidity_2m",
    "apparent_temperature", "precipitation",
    "wind_speed_10m", "shortwave_radiation",
]


def fetch_weather_hourly(lat, lon, date_str, timezone="Europe/Paris",
                         client=None, model="era5_land", date_end=None):
    """Fetch hourly weather data from Open-Meteo ERA5-Land."""
    if client is None:
        client = openmeteo_requests.Client()
    end_str = date_end if date_end is not None else date_str
    params = {
        "latitude": lat, "longitude": lon,
        "start_date": date_str, "end_date": end_str,
        "hourly": _HOURLY_VARS,
        "wind_speed_unit": "kmh",
        "timezone": timezone,
        "models": model,
    }
    responses = client.weather_api(_ARCHIVE_URL, params=params)
    r = responses[0]
    hourly = r.Hourly()
    times = pd.date_range(
        start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
        end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
        freq=pd.Timedelta(seconds=hourly.Interval()),
        inclusive="left",
    )
    def safe_var(i):
        """Extract variable i; return NaN array if unavailable."""
        try:
            arr = hourly.Variables(i).ValuesAsNumpy()
            return arr if arr is not None else np.full(len(times), np.nan)
        except Exception:
            return np.full(len(times), np.nan)
    df_w = pd.DataFrame({
        "time": times,
        "temperature_2m": safe_var(0),
        "relative_humidity_2m": safe_var(1),
        "apparent_temperature": safe_var(2),
        "precipitation": safe_var(3),
        "wind_speed_10m": safe_var(4),
        "shortwave_radiation": safe_var(5),
    })
    for col in df_w.columns:
        if col == "time":
            continue
        df_w[col] = pd.to_numeric(df_w[col], errors="coerce")
        df_w.loc[df_w[col] < -999, col] = np.nan
    return df_w


# Auto-detect race date from GPS timestamps
ts_start = df["timestamp"].iloc[0]
ts_end = df["timestamp"].iloc[-1]

# Handle timezone: assume naive = UTC, convert to target timezone for date
if ts_start.tzinfo is not None:
    race_start_local = ts_start.tz_convert(TIMEZONE)
    race_end_local = ts_end.tz_convert(TIMEZONE)
else:
    race_start_local = ts_start.tz_localize("UTC").tz_convert(TIMEZONE)
    race_end_local = ts_end.tz_localize("UTC").tz_convert(TIMEZONE)

race_date_str = race_start_local.strftime("%Y-%m-%d")
race_end_str = RACE_END if RACE_END is not None else race_end_local.strftime("%Y-%m-%d")

print(f"Race start (local): {race_start_local}")
print(f"Race end (local)  : {race_end_local}")
print(f"Fetching weather for: {race_date_str} -> {race_end_str}")

df_weather = fetch_weather_hourly(
    lat_center, lon_center, race_date_str,
    timezone=TIMEZONE, date_end=race_end_str
)
print(f"Weather data: {len(df_weather)} hourly points")
print(df_weather.head())


Race start (local): 2025-11-30 00:32:03+01:00
Race end (local)  : 2025-11-30 11:09:39+01:00
Fetching weather for: 2025-11-30 -> 2025-11-30
Weather data: 24 hourly points
                       time  temperature_2m  relative_humidity_2m  \
0 2025-11-29 22:00:00+00:00          2.9095             96.857986   
1 2025-11-29 23:00:00+00:00          3.3595             96.185455   
2 2025-11-30 00:00:00+00:00          3.8595             95.186836   
3 2025-11-30 01:00:00+00:00          4.2095             94.198547   
4 2025-11-30 02:00:00+00:00          4.3095             92.555710   

   apparent_temperature  precipitation  wind_speed_10m  shortwave_radiation  
0                   NaN            NaN             NaN                  NaN  
1                   NaN            NaN             NaN                  NaN  
2                   NaN            NaN             NaN                  NaN  
3                   NaN            NaN             NaN                  NaN  
4                   NaN  

---
## 3. Interpolate onto GPS DataFrame

In [ ]:
def make_utc_naive(ts):
    """Force any timestamp Series to UTC-naive for safe comparison."""
    if ts.dt.tz is not None:
        return ts.dt.tz_convert("UTC").dt.tz_localize(None)
    return ts


def enrich_df_with_weather(df, df_weather):
    """Interpolate hourly weather onto the GPS DataFrame."""
    # Normalize both timestamp columns to UTC-naive
    t_gps_naive = make_utc_naive(df["timestamp"])
    t_w_naive = make_utc_naive(df_weather["time"])

    # Debug
    print(f"  GPS range    : {t_gps_naive.iloc[0]} -> {t_gps_naive.iloc[-1]}")
    print(f"  Weather range: {t_w_naive.iloc[0]} -> {t_w_naive.iloc[-1]}")

    # Convert to float seconds for np.interp
    ref = pd.Timestamp("2000-01-01")
    t_gps_s = (t_gps_naive - ref).dt.total_seconds().to_numpy()
    t_w_s = (t_w_naive - ref).dt.total_seconds().to_numpy()

    print(f"  Epoch GPS    : {t_gps_s[0]:.0f} -> {t_gps_s[-1]:.0f}")
    print(f"  Epoch weather: {t_w_s[0]:.0f} -> {t_w_s[-1]:.0f}")

    if t_gps_s[0] > t_w_s[-1] or t_gps_s[-1] < t_w_s[0]:
        print("  ERROR: no overlap between GPS and weather timestamps!")
        return df

    for src_col, dst_col in [
        ("temperature_2m", "temp_api"),
        ("relative_humidity_2m", "humidity_api"),
        ("wind_speed_10m", "wind_kmh_api"),
        ("apparent_temperature", "apparent_temp_api"),
    ]:
        if src_col in df_weather.columns:
            vals = df_weather[src_col].to_numpy(dtype=float)
            if np.all(np.isnan(vals)):
                df[dst_col] = np.nan
                print(f"  {dst_col}: all NaN, skipped")
            else:
                df[dst_col] = np.interp(t_gps_s, t_w_s, vals)
                print(f"  {dst_col}: {df[dst_col].min():.2f} -> {df[dst_col].max():.2f}")
        else:
            df[dst_col] = np.nan

    return df


df = enrich_df_with_weather(df, df_weather)


  GPS range    : 2025-11-29 23:32:03 -> 2025-11-30 10:09:39
  Weather range: 2025-11-29 22:00:00 -> 2025-11-30 21:00:00
  Epoch GPS    : 817774323 -> 817812579
  Epoch weather: 817768800 -> 817851600
  temp_api: 3.63 -> 6.18
  humidity_api: 89.98 -> 95.65
  wind_kmh_api: all NaN, skipped
  apparent_temp_api: all NaN, skipped


---
## 4. Visualization: watch temp vs ERA5 + humidity/wind

In [ ]:
x_km = df["dist_m"] / 1000.0

fig_wx = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    vertical_spacing=0.10,
    subplot_titles=("Temperature: watch vs ERA5-Land",
                    "Humidity (%) and wind (km/h)")
)

# Panel 1: temperatures
if "temperature" in df.columns:
    # Watch temperature, smoothed over ~3 km
    win = max(1, int(3000 / max(1, df["dist_m"].diff().median())))
    # fig_wx.add_trace(go.Scatter(
    #     x=x_km,
    #     y=df["temperature"].rolling(win, min_periods=1).mean(),
    #     mode="lines", line=dict(color="darkorange", width=1.5),
    #     name="Watch temp (smoothed)",
    # ), row=1, col=1)

fig_wx.add_trace(go.Scatter(
    x=x_km, y=df["temp_api"],
    mode="lines", line=dict(color="steelblue", width=2),
    name="ERA5-Land temp",
), row=1, col=1)

fig_wx.add_trace(go.Scatter(
    x=x_km, y=df["apparent_temp_api"],
    mode="lines", line=dict(color="steelblue", width=1, dash="dash"),
    name="Apparent temp (ERA5)",
), row=1, col=1)

# Panel 2: humidity and wind
fig_wx.add_trace(go.Scatter(
    x=x_km, y=df["humidity_api"],
    mode="lines", line=dict(color="mediumseagreen", width=1.5),
    name="Humidity (%)",
), row=2, col=1)

fig_wx.add_trace(go.Scatter(
    x=x_km, y=df["wind_kmh_api"],
    mode="lines", line=dict(color="lightskyblue", width=1.5),
    name="Wind (km/h)",
), row=2, col=1)

fig_wx.update_yaxes(title_text="Temperature (C)", row=1, col=1)
fig_wx.update_yaxes(title_text="% / km/h", row=2, col=1)
fig_wx.update_xaxes(title_text="Distance (km)", row=2, col=1)
fig_wx.update_layout(
    template="plotly_dark", height=500,
    margin=dict(l=60, r=40, t=50, b=50),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)

if EXPORT_FOR_BLOG:
    fig_wx.write_html(
        os.path.join(EXPORT_DIR, "weather_along_race.html"),
        include_plotlyjs="cdn",
        full_html=True,
    )
    print("Exported: weather_along_race.html")
fig_wx.show()


Exported: weather_along_race.html


---
## Output summary

In [ ]:
if EXPORT_FOR_BLOG:
    for f in sorted(os.listdir(EXPORT_DIR)):
        print(f"  {f}")


  weather_along_race.html
